In [1]:
import subprocess

In [2]:
subprocess.run(["pip", "install", "pymupdf", "sentence-transformers", 
                "chromadb", "--quiet", "--break-system-packages"])

CompletedProcess(args=['pip', 'install', 'pymupdf', 'sentence-transformers', 'chromadb', '--quiet', '--break-system-packages'], returncode=0)

In [3]:
import fitz  # pymupdf
import chromadb
from sentence_transformers import SentenceTransformer
import re

print("Dependencies loaded")

/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies loaded


In [4]:
import urllib.request
import os

In [5]:
url = "https://www.apple.com/newsroom/pdfs/fy2023-q4/FY23_Q4_Consolidated_Financial_Statements.pdf"
filepath = "apple_q4_fy23.pdf"

In [6]:
if not os.path.exists(filepath):
    print("Downloading Apple Q4 FY23 financial statements...")
    urllib.request.urlretrieve(url, filepath)
    print(f"Downloaded: {filepath}")
else:
    print(f"Already exists: {filepath}")

Downloaded: apple_q4_fy23.pdf


In [7]:
# Check file size
size_kb = os.path.getsize(filepath) / 1024
print(f"File size: {size_kb:.1f} KB")

File size: 3913.3 KB


# Extract THe TExt 

In [8]:
def extract_text_from_pdf(filepath: str) -> list[dict]:
    """
    we used list of dict  because we want to track which  page each chunk came from for citations like "this answer came form page 4 of this doc"
    """
    doc = fitz.open(filepath)
    pages = []

    for page_num  in range(len(doc)):
        page = doc[page_num]

        #Extract raw text 
        text = page.get_text()

        #Clean the text 
        text = re.sub(r'\n+', '\n' , text) # Multiple newlines -> single
        text = re.sub(r' +', ' ', text) # Multiple spaces into single space 
        text = text.strip()

        if len(text) > 50:
            pages.append({
                "page_num": page_num + 1, 
                "text": text,
                "char_count": len(text)
            })

    doc.close()
    return pages

In [9]:
pages = extract_text_from_pdf("apple_q4_fy23.pdf")

In [10]:
print(f"Total pages extracted: {len(pages)}")
print(f"Total characters: {sum(p['char_count'] for p in pages):,}")
print(f"\nFirst page preview:")
print(pages[0]['text'][:500])
print("...")
print(f"\nPage 1 char count: {pages[0]['char_count']}")

Total pages extracted: 3
Total characters: 6,071

First page preview:
Apple Inc. 
CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (Unaudited) 
(In millions, except number of shares, which are reflected in thousands, and per-share amounts) 
 
Three Months Ended 
 
Twelve Months Ended 
 
September 30, 
2023 
 
September 24, 
2022 
 
September 30, 
2023 
 
September 24, 
2022 
Net sales: 
 
 
 
 
 Products 
! 
67,184 ! 
70,958 ! 
298,085 ! 
316,199 
 Services 
 
22,314 
19,188 
85,200 
78,129 
Total net sales (1) 
 
89,498 
90,146 
383,285 
394,328 
Cost of sales: 
 
...

Page 1 char count: 2205


In [11]:
# Download a real 10-K from SEC EDGAR — text-based, clean extraction

In [15]:
url = "https://www.sec.gov/Archives/edgar/data/320193/000032019323000106/aapl-20230930.htm"
filepath = "apple_10k_2023.htm"

In [16]:
# this will not work because we sec blocks requests without user agent header 

# if not os.path.exists(filepath):
#     print("Downloading Apple 10-K 2023 from SEC EDGAR...")
#     urllib.request.urlretrieve(url, filepath)
#     print(f"Downloaded: {filepath}")

# size_mb = os.path.getsize(filepath) / (1024 * 1024)
# print(f"File size: {size_mb:.1f} MB")

In [17]:
import requests

In [18]:
if not os.path.exists(filepath):
    print("Downloading Apple 10-K 2023 from SEC EDGAR...")
    # WHY User-Agent: SEC EDGAR requires it by law
    # they block bots without identification
    headers = {
        "User-Agent": "Sonu Verma sonuverma@email.com",
        "Accept-Encoding": "gzip, deflate",
        "Host": "www.sec.gov"
    }
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        with open(filepath, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded successfully")
    else:
        print(f"Failed: {response.status_code}")

size_mb = os.path.getsize(filepath) / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

Downloaded successfully
File size: 1.5 MB


# Extract Text from html 

In [19]:
from bs4 import BeautifulSoup

In [20]:
def extract_text_from_html(filepath: str) -> list[dict]:
    """
    beautiful soup parses html structure properly lets us target specific tags instead of raw text dump
    """

    with open(filepath,'r', encoding='utf-8' , errors='ignore') as f:
        html = f.read()

    soup = BeautifulSoup(html, 'html.parser')

    # Removing tags in html because it adds extra noise not content bb
    for tag in soup(['script', 'style', 'meta', 'link']):
        tag.decompose()

    # Every top level dv is a section html  doesnt have pages it have sections div and more
    sections = []

    # get all paragraph and table text
    elements = soup.find_all(['p','td','th','h1','h2','h3'])

    current_section = []
    current_length = 0
    section_num = 1

    for elem in elements:
        text = elem.get_text(strip=True)

        if len(text) <  10:
            continue

        current_section.append(text)
        current_length += len(text)

        if current_length > 2000:
            sections.append({
                "section_num": section_num,
                "text": " ".join(current_section),
                "char_count": current_length
            })
            section_num += 1
            current_section = []
            current_length = 0

    if current_section:
        sections.append({
            "section_num":section_num,
            "text": " ".join(current_section),
            "char_count": current_length
        })
    return sections

In [21]:
sections = extract_text_from_html("apple_10k_2023.htm")

In [22]:
print(f"Total sections extracted: {len(sections)}")
print(f"Total characters: {sum(s['char_count'] for s in sections):,}")
print(f"\nFirst section preview:")
print(sections[0]['text'][:500])

Total sections extracted: 13
Total characters: 26,413

First section preview:
California 94-2404110 (State or other jurisdictionof incorporation or organization) (I.R.S. Employer Identification No.) One Apple Park Way Cupertino,California (Address of principal executive offices) (Zip Code) Title of each class Trading symbol(s) Name of each exchange on which registered Common Stock, $0.00001 par value per share The Nasdaq Stock Market LLC 1.375% Notes due 2024 The Nasdaq Stock Market LLC 0.000% Notes due 2025 The Nasdaq Stock Market LLC 0.875% Notes due 2025 The Nasdaq Sto


In [23]:
# Inspect all sections to understand content quality
for i, section in enumerate(sections):
    print(f"Section {section['section_num']}: {section['char_count']} chars")
    print(f"  Preview: {section['text'][:150]}")
    print()

Section 1: 2051 chars
  Preview: California 94-2404110 (State or other jurisdictionof incorporation or organization) (I.R.S. Employer Identification No.) One Apple Park Way Cupertino,

Section 2: 2005 chars
  Preview: July 2, 2023 to August 5, 2023: Open market and privately negotiated purchases August 6, 2023 to September 2, 2023: August 2023 ASRs Open market and p

Section 3: 2016 chars
  Preview: Selling, general and administrative Total operating expenses Operating income Other income/(expense), net Income before provision for income taxes Pro

Section 4: 2010 chars
  Preview: Total shareholders’ equity, beginning balances Common stock and additional paid-in capital: Beginning balances Common stock issued Common stock withhe

Section 5: 2008 chars
  Preview: Services(2) Total net sales Numerator: Net income Denominator: Weighted-average basic shares outstanding 15,744,231 16,215,963 16,701,272 Effect of di

Section 6: 2003 chars
  Preview: Total other current liabilities Long-term 

In [25]:
def chunk_sections(sections: list[dict], min_quality_score: float = 0.3) -> list[dict]:
    """
    garbage chunks waste embedding space and corrupt  retrieval results with irrelevant content
    """

    financial_signals = [
        "revenue", "income", "profit", "loss", "sales", "earnings",
        "expense", "cost", "margin", "cash", "debt", "equity",
        "assets", "liabilities", "operating", "net", "gross", "total"
    ]

    chunks = []

    for section in sections:
        text_lower = section["text"].lower()

        # QUAlity score = ratio of financial signal words present 
        signals_found = sum(1 for word in financial_signals if word in text_lower)
        quality_score = signals_found / len(financial_signals)

        # 0.3 threshold : at least 30% of signal words must appear filters  boilerplate while keeping financial content
        if quality_score >= min_quality_score:
            chunks.append({
                "chunk_id": f"chunk_{section['section_num']:03d}",
                "text": section["text"],
                "char_count": section["char_count"],
                "quality_score": round(quality_score,3),
                "source": "apple_10k_2023",
                "section_num": section["section_num"]
            })

    return chunks

In [26]:
chunks = chunk_sections(sections)

In [28]:
print(f"Sections before filtering: {len(sections)}")
print(f"Chunks after filtering: {len(chunks)}")
print(f"\nChunks kept:")
for chunk in chunks:
    print(f"  {chunk['chunk_id']}: quality={chunk['quality_score']} | {chunk['text'][:100]}")

Sections before filtering: 13
Chunks after filtering: 6

Chunks kept:
  chunk_002: quality=0.611 | July 2, 2023 to August 5, 2023: Open market and privately negotiated purchases August 6, 2023 to Sep
  chunk_003: quality=0.722 | Selling, general and administrative Total operating expenses Operating income Other income/(expense)
  chunk_004: quality=0.722 | Total shareholders’ equity, beginning balances Common stock and additional paid-in capital: Beginnin
  chunk_005: quality=0.722 | Services(2) Total net sales Numerator: Net income Denominator: Weighted-average basic shares outstan
  chunk_006: quality=0.611 | Total other current liabilities Long-term taxes payable Other non-current liabilities Total other no
  chunk_007: quality=0.556 | Maturities(calendar year) Amount(in millions) EffectiveInterest Rate Amount(in millions) EffectiveIn


# Load embedding model

In [29]:
# MiniLm because it us 6x faaster than mpnet 

In [30]:
print("Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded")

Loading embedding model...


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4651.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded


In [32]:
# Peristent clinet because  we want the db to survive differnt restarts
client = chromadb.PersistentClient(path="./chroma_db")

In [33]:
try:
    client.delete_collection("financial_docs")
    print("Deleted existing collection")
except:
    pass

In [37]:
collection = client.create_collection(
    name="financial_docs",
    metadata={"hnsw:space": "cosine"}
)

InternalError: Collection [financial_docs] already exists

In [38]:
print("Collection created")

Collection created


In [39]:
# Embed and store all chunks
print(f"\nEmbedding {len(chunks)} chunks...")
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)


Embedding 6 chunks...


Batches: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.66it/s]


In [40]:
print(f"Embedding shape: {embeddings.shape}")

Embedding shape: (6, 384)


In [42]:
# Store embeddings in ChromaDB with metadata
collection.add(
    ids=[chunk["chunk_id"] for chunk in chunks],
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=[{
        "source": chunk["source"],
        "section_num": chunk["section_num"],
        "quality_score": chunk["quality_score"],
        "char_count": chunk["char_count"]
    } for chunk in chunks]
)
print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 6 chunks in ChromaDB


In [43]:
print("\nTest retrieval — 'Apple revenue and net income'")
results = collection.query(
    query_texts=["Apple revenue and net income"],
    n_results=3
)

print(f"\nTop 3 results:")
for i, (doc, metadata, distance) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0], 
    results["distances"][0]
)):
    print(f"\nResult {i+1}:")
    print(f"  Section: {metadata['section_num']} | Quality: {metadata['quality_score']} | Distance: {round(distance, 4)}")
    print(f"  Preview: {doc[:200]}")


Test retrieval — 'Apple revenue and net income'


/home/spectra/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 79.3M/79.3M [01:34<00:00, 877kiB/s]



Top 3 results:

Result 1:
  Section: 3 | Quality: 0.722 | Distance: 0.5567
  Preview: Selling, general and administrative Total operating expenses Operating income Other income/(expense), net Income before provision for income taxes Provision for income taxes Net income Earnings per sh

Result 2:
  Section: 2 | Quality: 0.611 | Distance: 0.5646
  Preview: July 2, 2023 to August 5, 2023: Open market and privately negotiated purchases August 6, 2023 to September 2, 2023: August 2023 ASRs Open market and privately negotiated purchases September 3, 2023 to

Result 3:
  Section: 6 | Quality: 0.611 | Distance: 0.659
  Preview: Total other current liabilities Long-term taxes payable Other non-current liabilities Total other non-current liabilities Interest and dividend income Interest expense Other income/(expense), net Tota


## RAG Ingestion Pipeline

### What I built
- PDF text extraction with PyMuPDF
- HTML extraction from SEC EDGAR with BeautifulSoup
- Quality filtering using financial signal words (0.3 threshold)
- Embedding with MiniLM (6 chunks × 384 dimensions)
- ChromaDB storage with metadata for citations

### Pipeline flow
PDF/HTML → Extract text → Clean → Section by 2000 chars → 
Quality filter → Embed → Store in ChromaDB

### Key decisions and why
- HTML over PDF for SEC filings: semantic structure preserved
- 2000 char sections: balance context vs chunk granularity
- Quality filter: removes boilerplate that corrupts retrieval
- Cosine distance: magnitude-independent similarity
- PersistentClient: survives restarts

### Problems found
- Dense embeddings alone insufficient: stock repurchase ranked #2 for revenue query
- Keyword quality scoring misses semantically similar boilerplate
- Fix: hybrid search (BM25 + dense) on Day 62

### Real numbers
- Apple 10-K: 1.5MB HTML, 13 sections, 6 chunks after filtering
- Embedding time: near instant for 6 chunks
- Top retrieval distance: 0.5567 (room for improvement)